In [ ]:
# ==========================================
# 1. SETUP & AUTOMATIC DOWNLOADS
# ==========================================
import os
import glob
import urllib.request
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import torch

# Download SAM checkpoint automatically if it doesn't exist locally
checkpoint_path = "sam_vit_b_01ec64.pth"
if not os.path.exists(checkpoint_path):
    print(f"Downloading SAM weights to {os.path.abspath(checkpoint_path)}...")
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    urllib.request.urlretrieve(url, checkpoint_path)
    print("Download complete!")

# ==========================================
# 2. CORE PIPELINE FUNCTIONS
# ==========================================

def get_sam_generator(checkpoint_path="sam_vit_b_01ec64.pth", model_type="vit_b"):
    """Initializes and returns the SAM automatic mask generator with tuned parameters."""
    if torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"
        
    print(f"Loading SAM model on {device.upper()}...")
    sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
    sam.to(device=device)
    
    # We explicitly pass parameters to force SAM to be a bit more 
    # cohesive and drop ultra-tiny pixel noise internally.
    return SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=32,
        pred_iou_thresh=0.86,
        stability_score_thresh=0.92,
        min_mask_region_area=200  # Automatically merge adjacent tiny regions
    )

def spatial_subdivide(mask_bool, n_splits=2):
    """
    Splits a large boolean mask along its longest spatial axis using PCA.
    Returns a list of new boolean masks.
    """
    coords = np.argwhere(mask_bool)
    if len(coords) < 10: 
        return [mask_bool]
        
    pca = PCA(n_components=1)
    projections = pca.fit_transform(coords)
    
    percentiles = np.linspace(0, 100, n_splits + 1)
    sub_masks = []
    
    for i in range(n_splits):
        lo = np.percentile(projections, percentiles[i])
        hi = np.percentile(projections, percentiles[i+1])
        
        if i == n_splits - 1:
            valid_idx = (projections >= lo) & (projections <= hi)
        else:
            valid_idx = (projections >= lo) & (projections < hi)
            
        valid_idx = valid_idx.flatten()
        subset_coords = coords[valid_idx]
        
        new_mask = np.zeros_like(mask_bool, dtype=bool)
        for (y, x) in subset_coords:
            new_mask[y, x] = True
            
        sub_masks.append(new_mask)
        
    return sub_masks

def process_regions(masks, img_shape, min_area_pct=0.005, max_area_pct=0.15, max_regions=12):
    """
    Filters, resolves overlaps, and aggressively culls the output 
    to keep only the top most significant regions for the VLM.
    """
    total_area = img_shape[0] * img_shape[1]
    
    # 1. Filter out noise (Raised slightly from 0.2% to 0.5%)
    filtered_masks = []
    for m in masks:
        area_pct = m['area'] / total_area
        if area_pct >= min_area_pct:
            filtered_masks.append((m['segmentation'], area_pct))
            
    # 2. Sort from SMALLEST to LARGEST (for bottom-up pixel claiming)
    filtered_masks.sort(key=lambda x: x[1], reverse=False)
    
    assigned_pixels = np.zeros(img_shape, dtype=bool)
    candidate_regions = []
    
    # 3. Bottom-up pixel claiming
    for mask_bool, _ in filtered_masks:
        valid_mask = mask_bool & (~assigned_pixels)
        valid_area_pct = valid_mask.sum() / total_area
        
        if valid_area_pct >= min_area_pct:
            if valid_area_pct > max_area_pct:
                # Subdivide large regions
                sub_masks = spatial_subdivide(valid_mask, n_splits=2)
                for sm in sub_masks:
                    if sm.sum() / total_area >= min_area_pct:
                        candidate_regions.append(sm)
                        assigned_pixels |= sm
            else:
                candidate_regions.append(valid_mask)
                assigned_pixels |= valid_mask
                
    # 4. NEW: Rank and Cull! 
    # Sort the resulting valid regions by their actual pixel area (Largest to Smallest)
    region_areas = [(region, region.sum()) for region in candidate_regions]
    region_areas.sort(key=lambda x: x[1], reverse=True)
    
    # Keep only the top `max_regions` 
    final_regions = [r[0] for r in region_areas[:max_regions]]
    
    print(f"Extracted {len(candidate_regions)} candidate regions -> Kept top {len(final_regions)} for VLM.")
    return final_regions

def compute_region_stats(regions, depth_map):
    """
    Computes mean, median, and std for each region based on the depth map.
    Handles regions with missing/invalid depth data gracefully.
    """
    region_stats = {}
    for i, region_mask in enumerate(regions):
        depth_pixels = depth_map[region_mask]
        
        # Filter out invalid depth pixels (<= 0 or NaN)
        valid_depths = depth_pixels[(depth_pixels > 0) & (~np.isnan(depth_pixels))]
        
        if len(valid_depths) == 0:
            # If a region (like the sky) has no valid depth in the ground truth,
            # assign NaN rather than skipping it, so indices stay aligned!
            region_stats[i] = {
                'mean': float('nan'),
                'median': float('nan'),
                'std': float('nan')
            }
        else:
            region_stats[i] = {
                'mean': float(np.mean(valid_depths)),
                'median': float(np.median(valid_depths)),
                'std': float(np.std(valid_depths))
            }
    return region_stats

def generate_vlm_annotation(image, regions):
    """
    Draws translucent colored masks and numbered centroids on the image.
    """
    annotated_image = image.copy()
    overlay = image.copy()
    
    np.random.seed(42) 
    
    for i, mask in enumerate(regions):
        color = np.random.randint(0, 255, size=(3,), dtype=np.uint8).tolist()
        overlay[mask] = color
        
        coords = np.argwhere(mask)
        if len(coords) > 0:
            y_c, x_c = coords.mean(axis=0).astype(int)
            text = str(i)
            font = cv2.FONT_HERSHEY_SIMPLEX
            cv2.putText(annotated_image, text, (x_c, y_c), font, 1.2, (0, 0, 0), 8, cv2.LINE_AA)
            cv2.putText(annotated_image, text, (x_c, y_c), font, 1.2, (255, 255, 255), 2, cv2.LINE_AA)

    cv2.addWeighted(overlay, 0.4, annotated_image, 0.6, 0, annotated_image)
    return annotated_image

# ==========================================
# 3. EXECUTION SCRIPT (Multi-Image Batch + Predictions)
# ==========================================

# Paths to your folders
data_dir = "train"
predictions_dir = "train_predicted_by_us"

# Find all RGB images in the folder
rgb_files = sorted(glob.glob(os.path.join(data_dir, "*_rgb.png")))

if not rgb_files:
    raise FileNotFoundError(f"Could not find any *_rgb.png files in '{data_dir}'.")

# Set how many images you want to test
num_examples = 10
examples_to_process = min(num_examples, len(rgb_files))

print(f"Found {len(rgb_files)} images. Running pipeline on the first {examples_to_process}...\n")

# Load SAM once outside the loop to save time
mask_generator = get_sam_generator(checkpoint_path)

for idx in range(examples_to_process):
    target_rgb_path = rgb_files[idx]
    target_depth_path = target_rgb_path.replace("_rgb.png", "_depth.npy")
    
    # NEW: Construct the path for the predicted depth map
    base_name = os.path.basename(target_rgb_path).replace("_rgb.png", "")
    target_pred_path = os.path.join(predictions_dir, f"{base_name}.npy")
    
    print("-" * 50)
    print(f"Processing Image {idx + 1}/{examples_to_process}: {base_name}")
    
    # 1. Load Real Image
    image_bgr = cv2.imread(target_rgb_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    H, W = image_rgb.shape[:2]

    # 2. Load Real Depth (Ground Truth) and Predicted Depth (D0)
    real_depth = np.load(target_depth_path)
    
    if not os.path.exists(target_pred_path):
        print(f"WARNING: Prediction file not found for {base_name}. Skipping...")
        continue
    pred_depth = np.load(target_pred_path)

    # 3. Run SAM
    raw_masks = mask_generator.generate(image_rgb)

    # 4. Filter and Subdivide
    final_regions = process_regions(raw_masks, img_shape=(H, W))

    # 5. Extract Stats (CRITICAL UPDATE: Now using pred_depth instead of real_depth!)
    stats = compute_region_stats(final_regions, pred_depth)
    print(f"Sample Region Stats (from Predicted Depth D0) for Image {idx + 1}:")
    for i in list(stats.keys())[:3]:
        print(f"  Region {i}: Mean Depth = {stats[i]['mean']:.2f}, Std = {stats[i]['std']:.2f}")

    # 6. Annotate Image for VLM
    annotated_img = generate_vlm_annotation(image_rgb, final_regions)

    # 7. Visualize Results Inline (Expanded to 1x4 Grid)
    # Increased figure width from 15 to 20 to fit the 4th image
    plt.figure(figsize=(20, 5))

    # Plot 1: Original
    plt.subplot(1, 4, 1)
    plt.title(f"Original Image ({idx+1})")
    plt.imshow(image_rgb)
    plt.axis('off')

    # Plot 2: Ground Truth (Enhanced)
    plt.subplot(1, 4, 2)
    plt.title("Ground Truth (Enhanced)")
    valid_pixels = real_depth[real_depth > 0]
    if len(valid_pixels) > 0:
        vmin_gt = np.percentile(valid_pixels, 2)
        vmax_gt = np.percentile(valid_pixels, 98)
    else:
        vmin_gt, vmax_gt = 0, 1
    vis_depth_gt = np.ma.masked_where(real_depth <= 0, real_depth)
    plt.imshow(vis_depth_gt, cmap='plasma', vmin=vmin_gt, vmax=vmax_gt)
    plt.axis('off')
    
    # Plot 3: Predicted Depth (NEW)
    plt.subplot(1, 4, 3)
    plt.title("Predicted Depth (D0)")
    # We apply the same percentile clipping to the prediction for a fair visual comparison
    valid_pred = pred_depth[pred_depth > 0]
    if len(valid_pred) > 0:
        vmin_pred = np.percentile(valid_pred, 2)
        vmax_pred = np.percentile(valid_pred, 98)
    else:
        vmin_pred, vmax_pred = 0, 1
    vis_depth_pred = np.ma.masked_where(pred_depth <= 0, pred_depth)
    plt.imshow(vis_depth_pred, cmap='plasma', vmin=vmin_pred, vmax=vmax_pred)
    plt.axis('off')

    # Plot 4: VLM Annotated
    plt.subplot(1, 4, 4)
    plt.title("VLM Annotated Regions")
    plt.imshow(annotated_img)
    plt.axis('off')

    plt.tight_layout()
    plt.show()